# Spaceborne SAR Ship Wake Analysis — High-Speed GPU Pipeline
## Explainable Physics-Guided SAR-Wake Geometry Fusion & Maneuver Recognition
Trains U-Net wake segmentation, Image baseline, Wake baseline, Geometry MLP, Visual-Geometry Fusion, and Physics-Guided regularized model on Kaggle GPU.

In [ ]:
# 1. Clone repository and change directory
!git clone https://github.com/mohaimin2103001/thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis

In [ ]:
# 2. Install dependencies
!pip install -q pyarrow scikit-image shapely pyyaml timm albumentations segmentation-models-pytorch onnx onnxruntime

In [ ]:
# 3. Create target directory structure
!mkdir -p data/raw data/processed data/interim checkpoints models outputs/figures outputs/metrics outputs/gradcam figures tables

In [ ]:
# 4. Fast download of authentic 1.46 GB UEIKAP SAR dataset from Zenodo (~20s on Kaggle)
!curl -L -C - "https://zenodo.org/records/18391027/files/UEIKAP_SAR.zip?download=1" -o data/raw/UEIKAP_SAR.zip

In [ ]:
# 5. Extract and normalize folder structure
!unzip -q -o data/raw/UEIKAP_SAR.zip -d data/raw/UEIKAP_SAR

import os, shutil
raw_dir = 'data/raw'
tgt_dir = os.path.join(raw_dir, 'UEIKAP_SAR')
for root, dirs, files in os.walk(raw_dir):
    if 'images' in dirs and root != tgt_dir:
        for d in dirs:
            src_p = os.path.join(root, d)
            dst_p = os.path.join(tgt_dir, d)
            if not os.path.exists(dst_p):
                shutil.move(src_p, dst_p)


In [ ]:
# 6. Rasterize official wake masks and derive telemetry ground truth (~25s)
!python -m src.data.ueikap_parser
!python -m src.data.build_master_dataset

In [ ]:
# 7. Train All 5 Research Models on GPU (~3-4 min)
# - Model 1: Image-Only Baseline (ResNet-34)
# - Model 2: Wake-Only Baseline (Masked SAR)
# - Model 3: Geometry-Only Baseline (MLP on Signed Curvatures)
# - Model 4: Deep Visual + Geometry Fusion Model
# - Model 5: Physics-Guided Fusion Model (ROT ~ v * kappa)
!python kaggle/KAGGLE_MASTER_TRAINING.py

In [ ]:
# 8. Run Master Scientific Evaluation, Counterfactuals & Publication Figures
!python -m src.evaluation.evaluate_all
!python -m src.evaluation.generate_publication_figures

In [ ]:
# 9. Package all results into thesis_results.zip
!zip -r /kaggle/working/thesis_results.zip checkpoints/ data/processed/ configs/ figures/ tables/
print('\n' + '=' * 65)
print('SUCCESS! Download thesis_results.zip from the Output panel on the right.')
print('=' * 65)